# 🌩️ Storm Prediction — Pipeline Front

Ce notebook lance le pipeline complet de `storm_prediction` :
1. **Vérification des dépendances**
2. **Configuration** (chemins, paramètres)
3. **Lancement** du batch ou d'un orage unique
4. **Résultats** : stats + lien vers `index.html`

## 1. Dépendances

In [70]:
import subprocess, sys

required = ["pandas", "numpy"]
for pkg in required:
    try:
        __import__(pkg)
        
        print(f"✅ {pkg}")
    except ImportError:
        print(f"📦 Installation de {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"✅ {pkg} installé")

✅ pandas
✅ numpy


## 2. Configuration

Modifie les variables ci-dessous selon tes besoins.

In [71]:
from pathlib import Path

# ─── Chemins ───────────────────────────────────────────────────────────────
SCRIPT_DIR  = Path("src/test_direction")
SCRIPT_PATH = SCRIPT_DIR / "storm_direction_analysis.py"
CSV_PATH    = Path("data/raw/data_with_storm_id.csv")
OUTPUT_DIR  = SCRIPT_DIR / "output"

# ─── Paramètres d'exécution ────────────────────────────────────────────────
STORM_ID       = None    # None = batch complet | ex: "Ajaccio_0068" = un seul orage
MIN_DURATION   = 1       # Durée minimale d'un orage en minutes
LIMIT          = 100     # None = tous les orages | int = limiter le batch

# ─── Vérifications ──────────────────────────────────────────────m──────────
print(f"Script   : {SCRIPT_PATH.resolve()}")
print(f"Script   : {'✅ trouvé' if SCRIPT_PATH.exists() else '❌ INTROUVABLE'}")

csv_resolved = CSV_PATH.resolve()
print(f"CSV      : {csv_resolved}")
print(f"CSV      : {'✅ trouvé' if csv_resolved.exists() else '❌ INTROUVABLE'}")

print(f"Output   : {OUTPUT_DIR.resolve()}")
mode = ("Orage unique → " + STORM_ID) if STORM_ID else ("Batch (" + (str(LIMIT) if LIMIT else "tous les orages") + ")")
print(f"Mode     : {mode}")


Script   : /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/src/test_direction/storm_direction_analysis.py
Script   : ✅ trouvé
CSV      : /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/data/raw/data_with_storm_id.csv
CSV      : ✅ trouvé
Output   : /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/src/test_direction/output
Mode     : Batch (100)


## 3. Lancement du pipeline

In [72]:
import subprocess, sys, time

cmd = [
    sys.executable,
    str(SCRIPT_PATH.resolve()),
    "--csv",    str(CSV_PATH.resolve()),
    "--output", str(OUTPUT_DIR.resolve()),
    "--min_duration", str(MIN_DURATION),
]

if STORM_ID:
    cmd += ["--storm_id", STORM_ID]
if LIMIT and not STORM_ID:
    cmd += ["--limit", str(LIMIT)]

print("🚀 Commande :", " ".join(cmd))
print("-" * 60)

t0 = time.time()
result = subprocess.run(
    cmd,
    cwd=str(SCRIPT_DIR.resolve()),
    capture_output=False,
    text=True
)

elapsed = time.time() - t0
print("-" * 60)
if result.returncode == 0:
    print(f"✅ Pipeline terminé en {elapsed:.1f}s")
else:
    print(f"❌ Erreur (code {result.returncode}) en {elapsed:.1f}s")


🚀 Commande : /home/alvi/Documents/deep learning/segmentation_env/bin/python /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/src/test_direction/storm_direction_analysis.py --csv /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/data/raw/data_with_storm_id.csv --output /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/src/test_direction/output --min_duration 1 --limit 100
------------------------------------------------------------
📂 Chargement de /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/data/raw/data_with_storm_id.csv...
⚡ 100 orages à analyser (durée >= 1.0 min)

  → /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/src/test_direction/output/Ajaccio_0002.html
  ✓ Ajaccio_0002: 42.4 km/h → E (25.35 km)
  ⏭️  Ajaccio_0006: pas assez d'éclairs (2)
  → /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/src/test_direction/output/Ajaccio_0007.html
  ✓ Ajaccio_0007

## 4. Résultats

In [73]:
import json
import pandas as pd
from IPython.display import display, HTML

# ─── Stats fichiers générés ────────────────────────────────────────────────
html_files = list(OUTPUT_DIR.glob("*.html"))
html_storms = [f for f in html_files if f.name != "index.html"]

print(f"📁 Fichiers générés dans {OUTPUT_DIR.resolve()}")
print(f"   {len(html_storms)} orages HTML + {'index.html ✅' if (OUTPUT_DIR / 'index.html').exists() else 'index.html ❌'}")

# ─── Lecture du JSON de résultats ─────────────────────────────────────────
json_path = OUTPUT_DIR / "storm_directions.json"
if json_path.exists():
    with open(json_path) as f:
        raw = json.load(f)

    # Supporte liste [{storm_id, ...}] ou dict {storm_id: {...}}
    if isinstance(raw, list):
        entries = [(item.get("storm_id", f"#{i}"), item) for i, item in enumerate(raw)]
    else:
        entries = list(raw.items())

    rows = []
    for storm_id, info in entries:
        r = info.get("ransac") or {}
        exit_p = r.get("predicted_exit") or {}
        conf = r.get("confidence")
        inlier = r.get("inlier_ratio")
        rows.append({
            "ID":           storm_id,
            "Aéroport":     info.get("airport", ""),
            "Éclairs":      info.get("n_lightnings", ""),
            "Centroïdes":   info.get("n_centroids", ""),
            "Direction":    r.get("direction", ""),
            "Vitesse km/h": round(r.get("speed_kmh", 0), 1) if r.get("speed_kmh") else None,
            "Conf %":       round(conf * 100) if conf is not None else None,
            "Inliers %":    round(inlier * 100) if inlier is not None else None,
        })

    df = pd.DataFrame(rows)
    # Forcer les colonnes numériques
    for col in ["Conf %", "Inliers %", "Vitesse km/h"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if not df.empty:
        df = df.sort_values("Conf %", ascending=False, na_position="last").reset_index(drop=True)
        print(f"\n📊 Top 10 par confiance RANSAC :")
        display(df.head(10))

        conf_vals = df["Conf %"].dropna()
        inlier_vals = df["Inliers %"].dropna()
        print(f"\n📈 Stats globales ({len(df)} orages) :")
        print(f"   Confiance moyenne : {conf_vals.mean():.1f}%")
        print(f"   Inliers moyens    : {inlier_vals.mean():.1f}%")
        print(f"   Conf ≥ 50%        : {(conf_vals >= 50).sum()} orages")
        print(f"   Inliers ≥ 80%     : {(inlier_vals >= 80).sum()} orages")
else:
    print("⚠️  storm_directions.json introuvable — lancez d'abord le pipeline (cellule 3)")


📁 Fichiers générés dans /home/alvi/Documents/Github/databattle2026-meteorage/storm_prediction/src/test_direction/output
   2246 orages HTML + index.html ✅

📊 Top 10 par confiance RANSAC :


,ID,Aéroport,Éclairs,Centroïdes,Direction,Vitesse km/h,Conf %,Inliers %
0,Ajaccio_0126,Ajaccio,,,SE,43.1,96.0,100.0
1,Ajaccio_0094,Ajaccio,,,NE,64.5,88.0,100.0
2,Ajaccio_0093,Ajaccio,,,NE,90.0,75.0,75.0
3,Ajaccio_0012,Ajaccio,,,NE,84.6,75.0,75.0
4,Ajaccio_0072,Ajaccio,,,E,40.7,73.0,80.0
5,Ajaccio_0041,Ajaccio,,,SO,9.4,68.0,100.0
6,Ajaccio_0089,Ajaccio,,,E,106.3,67.0,70.0
7,Ajaccio_0133,Ajaccio,,,SO,7.1,65.0,75.0
8,Ajaccio_0090,Ajaccio,,,SE,38.6,63.0,67.0
9,Ajaccio_0045,Ajaccio,,,SE,25.9,62.0,67.0



📈 Stats globales (90 orages) :
   Confiance moyenne : 34.8%
   Inliers moyens    : 52.4%
   Conf ≥ 50%        : 19 orages
   Inliers ≥ 80%     : 8 orages


In [75]:
import sys, subprocess
from IPython.display import display, HTML

index_path = OUTPUT_DIR / "index.html"
if index_path.exists():
    abs_path = index_path.resolve()

    # Commande d'ouverture selon l'OS
    if sys.platform == "darwin":
        open_cmd = f"open {abs_path}"
    elif sys.platform == "win32":
        open_cmd = f"start {abs_path}"
    else:
        open_cmd = f"xdg-open {abs_path}"

    display(HTML(f"""
    <div style="padding:12px; background:#1e1e2e; borderm-radius:8px; border:1px solid #313244;">
        <h3 style="color:#cdd6f4; margin:0 0 8px 0;">🗺️ Visualisation interactive</h3>
        <p style="color:#a6adc8; margin:0 0 12px 0;">Ouvre le fichier suivant dans ton navigateur :</p>
        <code style="background:#313244; color:#89b4fa; padding:8px 14px; border-radius:6px; display:block; font-size:13px;">
            {abs_path}
        </code>
        <p style="color:#a6adc8; margin:12px 0 0 0; font-size:12px;">
            💡 Ou depuis un terminal : <code style="color:#a6e3a1;">{open_cmd}</code>
        </p>
    </div>
    """))

    # Ouvrir automatiquement dans le navigateur
    if sys.platform == "darwin":
        subprocess.Popen(["open", str(abs_path)])
    elif sys.platform == "win32":
        subprocess.Popen(["start", str(abs_path)], shell=True)
    else:
        subprocess.Popen(["xdg-open", str(abs_path)])
else:
    print("⚠️  index.html non trouvé — lancez d'abord le pipeline (cellule 3)")


Gtk-Message: 16:57:29.236: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.
